In [ ]:
# pip install beeai-framework

In [ ]:
# pip install "beeai-framework[a2a]"

In [ ]:
# pip install a2a-sdk mcp openai

In [ ]:
# pip install orjson

# Creating an Agentic multi-agent system using A2A

In this final coding exercise, you will create a comprehensive "Healthcare Concierge" system. You will use the **BeeAI Framework** to orchestrate all three agents you have built so far (Policy, Research, and Provider). You will create a custom router that will decide which A2A agent to hand off to based on the user's complex query.

In [ ]:
import asyncio
import os
from typing import Any

from IPython.display import Markdown, display
from dotenv import load_dotenv
from gates_openai import assign_route



## Define BeeAI Components

Here you will:
1.  Import BeeAI framework components, including `RequirementAgent` and `HandoffTool`.
2.  Define `A2AAgent` instances for each of your running servers.
3.  Use `check_agent_exists()` to fetch the metadata (AgentCard) from each server.

In [ ]:
from beeai_framework.adapters.a2a.agents import A2AAgent
# from beeai_framework.adapters.vertexai import VertexAIChatModel
from beeai_framework.adapters.openai import OpenAIChatModel
from beeai_framework.agents.requirement import RequirementAgent
from beeai_framework.agents.requirement.requirements.conditional import (
    ConditionalRequirement,
)
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.memory.unconstrained_memory import UnconstrainedMemory
from beeai_framework.middleware.trajectory import EventMeta, GlobalTrajectoryMiddleware
from beeai_framework.tools import Tool
from beeai_framework.tools.handoff import HandoffTool
from beeai_framework.tools.think import ThinkTool


class ConciseGlobalTrajectoryMiddleware(GlobalTrajectoryMiddleware):
    def _format_prefix(self, meta: EventMeta) -> str:
        prefix = super()._format_prefix(meta)
        return prefix.rstrip(": ")

    def _format_payload(self, value: Any) -> str:
        return ""

In [ ]:
host = "localhost"
policy_agent_port = 9999
research_agent_port = 9998
provider_agent_port = 9997
healthcare_agent_port = 9996

In [ ]:
policy_agent = A2AAgent(
    url=f"http://{host}:{policy_agent_port}", 
    memory=UnconstrainedMemory()
)
# Run `check_agent_exists()` to fetch and populate AgentCard
await policy_agent.check_agent_exists()
print("\tℹ️", f"{policy_agent.name} initialized")

In [ ]:
research_agent = A2AAgent(
    url=f"http://{host}:{research_agent_port}", 
    memory=UnconstrainedMemory()
)
await research_agent.check_agent_exists()
print("\tℹ️", f"{research_agent.name} initialized")

In [ ]:
provider_agent = A2AAgent(
    url=f"http://{host}:{provider_agent_port}", 
    memory=UnconstrainedMemory()
)
await provider_agent.check_agent_exists()
print("\tℹ️", f"{provider_agent.name} initialized")

## Configure the Orchestrator (Healthcare Concierge)

In [ ]:
healthcare_agent = RequirementAgent(
    name="Healthcare Agent",
    description="""A personal concierge for Healthcare Information, 
    customized to your policy.""",
    llm=OpenAIChatModel(
        model_id="gpt-4o-mini",
        api_key=assign_route(),
        #base_url="http://172.16.0.237:11434/v1",
    ),
    tools=[
        thinktool := ThinkTool(),
        policy_tool := HandoffTool(
            target=policy_agent,
            name=policy_agent.name,
            description=policy_agent.agent_card.description,
        ),
        research_tool := HandoffTool(
            target=research_agent,
            name=research_agent.name,
            description=research_agent.agent_card.description,
        ),
        provider_tool := HandoffTool(
            target=provider_agent,
            name=provider_agent.name,
            description=provider_agent.agent_card.description,
        ),
    ],
    requirements=[
        # ConditionalRequirement(policy_tool, consecutive_allowed=False),
        ConditionalRequirement(
            thinktool, force_at_step=1, force_after=Tool, 
            consecutive_allowed=False
        ),
    ],
    role="Healthcare Concierge",
    instructions=(
        f"""You are a concierge for healthcare services. Your task is 
        to handoff to one or more agents to answer questions and provide 
        a detailed summary of their answers. Be sure that all of their 
        questions are answered before responding.
        Use `{policy_agent.name}` to answer insurance-related questions.
        
        IMPORTANT: When returning answers about providers, only output 
        providers from `{provider_agent.name}` and only provide insurance 
        information based on the results from `{policy_agent.name}`.

        In your output, put which agent gave you the information!"""
    ),
)

print("\tℹ️", f"{healthcare_agent.meta.name} initialized")

## Run the Full Workflow

Test the system with a complex query that requires information from all three sub-agents.

In [ ]:
try:
    response = await healthcare_agent.run(
        """I'm based in Caloocan, Metro Manila. How do I get mental health therapy near me 
        and what does my insurance cover?"""
    ).middleware(ConciseGlobalTrajectoryMiddleware())
except Exception as e:
    import traceback
    traceback.print_exc()
display(Markdown(response.last_message.text))

## Write the Agent Code to a File

In [ ]:
%%writefile a2a_healthcare_agent.py
from typing import Any
import asyncio
from beeai_framework.adapters.a2a.serve.server import A2AServer, A2AServerConfig
from beeai_framework.adapters.a2a.agents import A2AAgent
from beeai_framework.adapters.openai import OpenAIChatModel
from beeai_framework.agents.requirement import RequirementAgent
from beeai_framework.agents.requirement.requirements.conditional import ConditionalRequirement
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.memory.unconstrained_memory import UnconstrainedMemory
from beeai_framework.middleware.trajectory import EventMeta, GlobalTrajectoryMiddleware
from beeai_framework.serve.utils import LRUMemoryManager
from beeai_framework.tools import Tool, tool
from beeai_framework.tools.handoff import HandoffTool
from beeai_framework.tools.think import ThinkTool
from gates_openai import assign_route

# Log only tool calls
class ConciseGlobalTrajectoryMiddleware(GlobalTrajectoryMiddleware):
    def _format_prefix(self, meta: EventMeta) -> str:
        prefix = super()._format_prefix(meta)
        return prefix.rstrip(": ")

    def _format_payload(self, value: Any) -> str:
        return ""

def main():
    print(f"Running A2A Orchestrator Agent")

    host = "localhost"
    policy_agent_port = 9999
    research_agent_port = 9998
    provider_agent_port = 9997
    healthcare_agent_port = 9996

    # Log only tool calls
    GlobalTrajectoryMiddleware(target=[Tool]) 

    policy_agent = A2AAgent(
        url=f"http://{host}:{policy_agent_port}", memory=UnconstrainedMemory()
    )
    # Run `check_agent_exists()` to fetch and populate AgentCard
    asyncio.run(policy_agent.check_agent_exists())
    print("\tℹ️", f"{policy_agent.name} initialized")
    
    research_agent = A2AAgent(
        url=f"http://{host}:{research_agent_port}", memory=UnconstrainedMemory()
    )
    asyncio.run(research_agent.check_agent_exists())
    print("\tℹ️", f"{research_agent.name} initialized")

    provider_agent = A2AAgent(
        url=f"http://{host}:{provider_agent_port}", memory=UnconstrainedMemory()
    )
    asyncio.run(provider_agent.check_agent_exists())
    print("\tℹ️", f"{provider_agent.name} initialized")

    healthcare_agent = RequirementAgent(
        name="Healthcare Agent",
        description="A personal concierge for Healthcare Information, customized to your policy.",
        llm=OpenAIChatModel(
            model_id="gpt-4o-mini",
            api_key=assign_route(),
            #base_url="http://172.16.0.237:11434/v1",
        ),
        tools=[
            thinktool:=ThinkTool(),
            policy_tool:=HandoffTool(
                target=policy_agent,
                name=policy_agent.name,
                description=policy_agent.agent_card.description,
            ),
            research_tool:=HandoffTool(
                target=research_agent,
                name=research_agent.name,
                description=research_agent.agent_card.description,
            ),
            provider_tool:=HandoffTool(
                target=provider_agent,
                name=provider_agent.name,
                description=provider_agent.agent_card.description,
            ),
        ],
        requirements=[
            ConditionalRequirement(policy_tool, consecutive_allowed=False),
            ConditionalRequirement(thinktool, force_at_step=1, force_after=Tool, consecutive_allowed=False),
        ],
        role="Healthcare Concierge",
        instructions=(
            f"""You are a concierge for healthcare services. Your task is to handoff to one or more agents to answer questions and provide a detailed summary of their answers. Be sure that all of their questions are answered before responding.
            Use `{policy_agent.name}` to answer insurance-related questions.
            
            IMPORTANT: When returning answers about providers, only output providers from `{provider_agent.name}` and only provide insurance information based on the results from `{policy_agent.name}`.
    
            In your output, put which agent gave you the information!"""
        ),
    )

    print("\tℹ️", f"{healthcare_agent.meta.name} initialized")
    

### Add the A2AServer registration
The only change to run an agent as an A2A agent is to add the single A2AServer registration statement.

In [ ]:
%%writefile a2a_healthcare_agent.py -a

    # Register the agent with the A2A server and run the HTTP server
    # we use LRU memory manager to keep limited amount of sessions in the memory
    A2AServer(
        config=A2AServerConfig(port=healthcare_agent_port, protocol="jsonrpc", host=host ),
        memory_manager=LRUMemoryManager(maxsize=100),
    ).register(healthcare_agent, send_trajectory=True).serve()

if __name__ == "__main__":
    main()   

## Serve the Concierge Agent

Finally, you can register this high-level "Concierge" agent itself as an A2A server. This demonstrates the recursive power of A2A: an agent composed of other A2A agents can itself be exposed as an A2A agent.

Now to activate your configured A2A agent, you would need to run your agent server. You can run the agent server using `uv`:

- Open Terminal
- `uv init`
- `uv venv`
- `source .venv/bin/activate`
- `uv add "beeai-framework[a2a]" a2a-sdk mcp openai orjson`
- Type `uv run a2a_healthcare_agent.py` to run the server and activate your A2A agent.


## Run the Client

Question: I'm based in Caloocan, Metro Manila. How do I get mental health therapy near me and what does my insurance cover?


In [ ]:
agent = A2AAgent(url="http://localhost:9996", 
                 memory=UnconstrainedMemory())
response = await agent.run(
    "I'm based in Caloocan, Metro Manila. How do I get mental health therapy near me and what does my insurance cover?"
).middleware(ConciseGlobalTrajectoryMiddleware())
display(Markdown(response.last_message.text))

## Resources

- [BeeAI Framework](https://framework.beeai.dev/introduction/welcome)
- [BeeAI Requirement Agent](https://framework.beeai.dev/modules/agents/requirement-agent)
- [BeeAI Framework GitHub](https://github.com/i-am-bee/beeai-framework)